# Collecte GTFS : arrêts, lignes et horaires des transports en commun

Charge le catalogue des GTFS publiés, l'agrégat national de tous les arrêts de France, puis les GTFS complets (lignes, courses, horaires) des réseaux de la zone d'étude.

**Prérequis** : un Lakehouse attaché comme Lakehouse par défaut du notebook.
Les fichiers bruts sont écrits dans `Files/raw/`, les tables Delta dans `Tables/`.

*Notebook généré depuis `src/ingestion/` par `fabric/build_notebooks.py` : modifier les scripts, pas ce notebook.*

In [ ]:
# Zone d'étude : Bordeaux Métropole (TBM) + réseaux de Gironde
ZONE = ["Bordeaux Métropole", "Gironde", "Aéroport de Bordeaux"]  # ["*"] = toute la France
EXCLURE_AGREGATS = True  # écarte les agrégats régionaux qui dupliquent les réseaux locaux
AVEC_STOP_TIMES = True  # horaires détaillés (volumineux)
FORCE = False


## Outils communs

In [ ]:
"""Outils communs à la collecte : chemins, téléchargements HTTP, écriture des tables.

Les mêmes scripts tournent en local (sortie dans data/raw/) et dans un notebook
Microsoft Fabric (sortie dans le Lakehouse attaché : /lakehouse/default/Files/raw/).
"""
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

FABRIC_LAKEHOUSE = Path("/lakehouse/default/Files/raw")
LOCAL_RAW = Path(__file__).resolve().parents[2] / "data" / "raw" if "__file__" in globals() else Path("data/raw")

USER_AGENT = "hackathon-mobility-citiz/1.0 (collecte open data)"
PAN_API = "https://transport.data.gouv.fr/api/datasets"


def raw_dir() -> Path:
    """Dossier racine des données brutes : Lakehouse Fabric si présent, sinon data/raw/."""
    base = Path(os.environ["RAW_DIR"]) if "RAW_DIR" in os.environ else (
        FABRIC_LAKEHOUSE if FABRIC_LAKEHOUSE.parent.exists() else LOCAL_RAW
    )
    base.mkdir(parents=True, exist_ok=True)
    return base


def tables_dir() -> Path:
    """Tables Parquet prêtes à être chargées en tables Delta dans Fabric."""
    d = raw_dir() / "tables"
    d.mkdir(parents=True, exist_ok=True)
    return d


def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def http_get(url: str, timeout: int = 60, retries: int = 3, **kwargs) -> requests.Response:
    headers = {"User-Agent": USER_AGENT, **kwargs.pop("headers", {})}
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def get_json(url: str, timeout: int = 30):
    return http_get(url, timeout=timeout).json()


def download(url: str, dest: Path, force: bool = False) -> Path:
    """Télécharge un fichier en streaming. Ne retélécharge pas s'il existe déjà."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and not force:
        print(f"  déjà présent : {dest.name}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with http_get(url, timeout=300, stream=True) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    print(f"  téléchargé : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo)")
    return dest


def save_json(obj, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(obj, ensure_ascii=False))


def _manquant(v) -> bool:
    """Valeur manquante (None, NaN, pd.NA), sans échouer sur les listes ou dictionnaires des flux JSON."""
    return v is None or v is pd.NA or (isinstance(v, float) and v != v)


def _types_homogenes(df: pd.DataFrame) -> pd.DataFrame:
    """Les flux open data mélangent parfois nombres et textes dans une même colonne :
    on convertit ces colonnes en texte pour que Parquet / Delta les acceptent.
    Les colonnes booléennes avec des valeurs manquantes restent booléennes (type nullable)."""
    for col in df.columns[df.dtypes == object]:
        valeurs = df[col].dropna()
        types = {type(v) for v in valeurs}
        if not types:  # colonne entièrement vide : type "null" refusé par Delta, on la type en texte
            df[col] = df[col].astype("string")
        elif types & {bool, np.bool_} and types <= {bool, np.bool_, str} and valeurs.map(
                lambda v: str(v) in ("True", "False")).all():
            # booléens avec des trous (ou déjà écrits en texte "True"/"False" dans l'historique)
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v) == "True").astype("boolean")
        elif len(types) > 1 or types - {str}:
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v))
    return df


def write_table(df: pd.DataFrame, name: str, append: bool = False) -> Path:
    """Écrit une table Parquet dans <raw>/tables/<name>.parquet (ou ajoute à l'existante)."""
    path = tables_dir() / f"{name}.parquet"
    df = _types_homogenes(df.copy())  # ne modifie pas le DataFrame de l'appelant
    if append and path.exists():
        df = _types_homogenes(pd.concat([pd.read_parquet(path), df], ignore_index=True))
    df.to_parquet(path, index=False)
    print(f"  table {name} : {len(df):,} lignes -> {path}")
    return path


def pan_catalog() -> list:
    """Catalogue complet du Point d'Accès National transport.data.gouv.fr."""
    return get_json(PAN_API, timeout=120)


## Collecte

In [ ]:
"""Collecte GTFS : arrêts, lignes et horaires des transports en commun.

Sources :
  - Catalogue du Point d'Accès National : https://transport.data.gouv.fr/datasets?type=public-transit
  - Agrégat national des arrêts : https://www.data.gouv.fr/datasets/arrets-de-transport-en-france

Trois niveaux de collecte :
  1. le catalogue de tous les GTFS publiés (léger) ;
  2. tous les arrêts de France en un seul fichier (environ 440 Mo, non dédoublonné) ;
  3. les GTFS complets (lignes, courses, horaires) des réseaux de la zone d'étude.

Sorties :
  <raw>/gtfs/arrets_france.csv
  <raw>/gtfs/zips/<jeu>_<ressource>.zip
  <raw>/tables/gtfs_catalogue.parquet
  <raw>/tables/gtfs_arrets_france.parquet
  <raw>/tables/gtfs_ressources.parquet  (statut de lecture de chaque GTFS de la zone)
  <raw>/tables/gtfs_<fichier>.parquet   (agency, routes, stops, trips, calendar, calendar_dates, stop_times)
"""
import zipfile

import pandas as pd


ARRETS_FRANCE_DATASET = "https://www.data.gouv.fr/api/1/datasets/arrets-de-transport-en-france/"
# Zone d'étude retenue : Bordeaux Métropole (TBM) + réseaux de Gironde pour le périurbain
ZONE_BORDEAUX = ["Bordeaux Métropole", "Gironde", "Aéroport de Bordeaux"]
FICHIERS_GTFS = ["agency", "routes", "stops", "trips", "calendar", "calendar_dates", "stop_times"]


def catalogue() -> pd.DataFrame:
    rows = []
    for ds in pan_catalog():
        if ds.get("type") != "public-transit":
            continue
        zones = ds.get("covered_area") or []
        for r in ds.get("resources", []):
            if r.get("format") != "GTFS":
                continue
            meta = r.get("metadata") or {}
            rows.append({
                "jeu_id": ds["id"],
                "jeu_slug": ds["slug"],
                "jeu_titre": ds["title"],
                "producteur": (ds.get("publisher") or {}).get("name"),
                "zones_couvertes": " | ".join(z.get("nom", "") for z in zones),
                "zones_insee": " | ".join(str(z.get("insee", "")) for z in zones),
                "types_zones": " | ".join(z.get("type", "") for z in zones),
                "ressource_id": r.get("id"),
                "ressource_titre": r.get("title"),
                "modes": ",".join(meta.get("modes") or []),
                "reseaux": ",".join(meta.get("networks") or []),
                "date_fin_validite": meta.get("end_date"),
                "disponible": r.get("is_available"),
                "url": r.get("url"),
                "page_pan": ds.get("page_url"),
            })
    df = pd.DataFrame(rows)
    df["date_collecte"] = now_utc()
    write_table(df, "gtfs_catalogue")
    return df


def arrets_france(force: bool = False) -> pd.DataFrame:
    """Agrégat national de tous les arrêts GTFS (dernière version publiée sur data.gouv.fr)."""
    meta = get_json(ARRETS_FRANCE_DATASET)
    ressource = next(r for r in meta["resources"] if r["format"] == "csv")
    print(f"  agrégat national : {ressource['title']}")
    chemin = download(ressource["url"], raw_dir() / "gtfs" / "arrets_france.csv", force=force)
    df = pd.read_csv(chemin, dtype=str, low_memory=False)
    for col in ("stop_lat", "stop_lon"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fichier_source"] = ressource["title"]
    df["date_collecte"] = now_utc()
    write_table(df, "gtfs_arrets_france")
    return df


def selectionner(cat: pd.DataFrame, zone: list[str], exclure_agregats: bool = True) -> pd.DataFrame:
    """Filtre le catalogue sur des mots-clés (nom de zone, de réseau, titre ou code INSEE).
    zone = ["*"] sélectionne tous les GTFS de France.
    exclure_agregats : écarte les agrégats régionaux, qui dupliquent les réseaux locaux."""
    cat = cat[cat["disponible"].astype("boolean").fillna(False).astype(bool)]
    if exclure_agregats:
        cat = cat[~cat["jeu_titre"].str.lower().str.startswith("agrégat")]
    if zone == ["*"]:
        return cat
    texte = (cat["jeu_titre"] + " " + cat["zones_couvertes"] + " " + cat["zones_insee"] + " "
             + cat["reseaux"] + " " + cat["producteur"].fillna("")).str.lower()
    masque = pd.Series(False, index=cat.index)
    for mot in zone:
        masque |= texte.str.contains(mot.lower(), regex=False)
    return cat[masque]


def gtfs_complets(cat: pd.DataFrame, zone: list[str], avec_stop_times: bool = True, force: bool = False,
                  exclure_agregats: bool = True) -> None:
    choix = selectionner(cat, zone, exclure_agregats)
    print(f"  {len(choix)} GTFS sélectionnés pour la zone {zone}")
    fichiers = FICHIERS_GTFS if avec_stop_times else [f for f in FICHIERS_GTFS if f != "stop_times"]
    morceaux = {f: [] for f in fichiers}
    statuts = []
    for r in choix.to_dict("records"):
        zip_path = raw_dir() / "gtfs" / "zips" / f"{r['jeu_slug']}_{r['ressource_id']}.zip"
        lus = {}
        try:
            download(r["url"], zip_path, force=force)
            with zipfile.ZipFile(zip_path) as z:
                noms = {n.split("/")[-1]: n for n in z.namelist()}
                for f in fichiers:
                    if f"{f}.txt" in noms:
                        with z.open(noms[f"{f}.txt"]) as fh:
                            df = pd.read_csv(fh, dtype=str, encoding="utf-8-sig")
                        df.insert(0, "ressource_id", str(r["ressource_id"]))
                        df.insert(1, "jeu_titre", r["jeu_titre"])
                        lus[f] = df
            statut = "ok"
        except Exception as e:
            print(f"  ! {r['jeu_titre']} ({r['ressource_id']}) ignoré : {e}")
            statut, lus = f"erreur: {e}", {}
        # une ressource n'est publiée que si tous ses fichiers ont été lus : pas d'arrêts sans courses
        for f, df in lus.items():
            morceaux[f].append(df)
        statuts.append({"ressource_id": str(r["ressource_id"]), "jeu_titre": r["jeu_titre"], "url": r["url"],
                        "statut": statut, "fichiers": ",".join(lus),
                        "nb_arrets": len(lus["stops"]) if "stops" in lus else None})
    horodatage = now_utc()
    colonnes = ["ressource_id", "jeu_titre", "url", "statut", "fichiers", "nb_arrets"]
    write_table(pd.DataFrame(statuts, columns=colonnes).assign(date_collecte=horodatage), "gtfs_ressources")
    for f, dfs in morceaux.items():
        if dfs:
            df = pd.concat(dfs, ignore_index=True).assign(date_collecte=horodatage)
            for col in ("stop_lat", "stop_lon"):
                if col in df:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
            write_table(df, f"gtfs_{f}")


def collecter(zone: list[str], avec_stop_times: bool = True, force: bool = False,
              exclure_agregats: bool = True) -> None:
    print("GTFS - transports en commun")
    cat = catalogue()
    arrets_france(force=force)
    gtfs_complets(cat, zone, avec_stop_times=avec_stop_times, force=force, exclure_agregats=exclure_agregats)


In [ ]:
collecter(zone=ZONE, avec_stop_times=AVEC_STOP_TIMES, force=FORCE, exclure_agregats=EXCLURE_AGREGATS)


## Chargement en tables Delta

In [ ]:
TABLES = ['gtfs_catalogue', 'gtfs_arrets_france', 'gtfs_ressources', 'gtfs_agency', 'gtfs_routes', 'gtfs_stops', 'gtfs_trips', 'gtfs_calendar', 'gtfs_calendar_dates', 'gtfs_stop_times']

# Chargement des fichiers Parquet (Files/raw/tables) en tables Delta du Lakehouse par défaut.
# L'écriture passe par l'adresse OneLake (abfss) : le dossier monté /lakehouse/default/Tables
# interrompt les envois de gros fichiers.
from pathlib import Path
import pyarrow.parquet as pq
from deltalake import write_deltalake

try:
    import notebookutils
    ctx = notebookutils.runtime.context
    TABLES_DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
                    f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
    OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}
except ImportError:  # hors Fabric : tables Delta locales, pour tester
    TABLES_DELTA, OPTIONS = str(raw_dir() / "delta"), None

for nom in TABLES:
    source = raw_dir() / "tables" / f"{nom}.parquet"
    if not source.exists():
        print(f"{nom} : absent, ignoré")
        continue
    write_deltalake(f"{TABLES_DELTA}/{nom}", pq.read_table(source), mode="overwrite",
                    schema_mode="overwrite", storage_options=OPTIONS)
    print(f"{nom} : table Delta créée")
